In [4]:
from google.colab import userdata
import os

os.environ['GITHUB_PAT'] = userdata.get('GITHUB_PAT')

!git clone https://github.com/navidivan/rsm.git
%cd rsm

Cloning into 'rsm'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 52 (delta 11), reused 48 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 12.09 MiB | 17.38 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/rsm


In [8]:
# Login (Required once)
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [5]:
!pip install -q hydra-core wandb pydantic ninja argdantic coolname huggingface_hub adam-atan2 einops

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 19.9 MB/s eta 0:00:00


In [6]:
!python dataset/build_sudoku_dataset.py --output-dir data/sudoku-extreme-1k-aug-all

train.csv: 100% 719M/719M [00:02<00:00, 280MB/s]
100% 3831994/3831994 [00:01<00:00, 2081647.45it/s]
test.csv: 100% 79.4M/79.4M [00:01<00:00, 66.8MB/s]
100% 422786/422786 [00:00<00:00, 1899869.95it/s]


In [ ]:
# Define run name. prob_detach_prev_H 0 will always train the last 2 steps
run_name="Sudoku_Curriculum_Run"

!python pretrain.py \
arch=rsm \
data_paths="[data/sudoku-extreme-1k-aug-all]" \
evaluators="[]" \
epochs=2 \
eval_interval=7484 \
lr=1e-3 puzzle_emb_lr=1e-4 weight_decay=.1 puzzle_emb_weight_decay=1 global_batch_size=512 \
arch.mlp_t=True arch.pos_encodings=none \
arch.L_layers=1 \
arch.H_cycles=2 arch.L_cycles=2 \
+arch.prob_detach_prev_H=0.99 \
+project_name="Sudoku_Extreme" \
+curriculum_milestones="[20.1, 40.1, 55.1, 65.1, 75.1, 85.1]" \
+curriculum_heads_to_add="[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]" \
+curriculum_L_milestones="[30.1, 50.1, 70.1, 80.1, 84.1, 86.1, 88.1, 90.1, 92.1, 94.1]" \
+curriculum_L_to_add="[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]" \
+arch.freeze_emb_prc=100.0 \
+grad_clip_norm=1.0 \
+local_warmup_steps=100 \
+optimizer_reset_scale=1 \
+arch.alpha_blend=False \
+save_checkpoints_at_percent="[10, 20, 30, 40, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 99, 99.9]" \
+short_eval_interval=500 \
+short_eval_fraction=0.005 \
+run_name=${run_name} ema=True

# Inferance

In [ ]:
import os
import sys
import yaml
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader

# Make repo imports work
sys.path.append(os.getcwd())

from models.recursive_reasoning.grm import GRM  # expects GRM(config_dict)

# =========================
# USER SETTINGS
# =========================
CHECKPOINT_DIR = "/content/drive/MyDrive/TinyRecursiveRuns/Sudoku_Extreme/_2026-02-14_23-39-44/"
WEIGHTS_FILE   = "ckpt_pct_99.pth"

DATA_PATH   = "data/sudoku-extreme-1k-aug-all/test"
NUM_SAMPLES = 2048
EVAL_BATCH  = 512

# Format: (H_cycles, L_cycles)
# You can mix and match to see if depth (H) or width (L) matters more.
TEST_CONFIGS = [
    (1, 8),   # Baseline L
    (1, 15),
    (1, 20),  #
    (2, 8),
    (2, 15),   #
    (2, 20),   #
    (3, 8),
    (3, 15),   #
    (3, 20),   #
    (10, 2),
    (10, 3),   #
    (10, 20),   #
    (20, 2),
    (20, 3),   #
    (20, 12),   #
    (20, 20),   #
    (40, 12),   #
    (40, 30),   #
    (320, 12),   #
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IGNORE_LABEL_ID = -100


# =========================
# HELPERS
# =========================
def strip_prefixes(state_dict: dict) -> dict:
    """Remove common wrappers: DDP 'module.' and LossHead 'model.'."""
    out = {}
    for k, v in state_dict.items():
        if k.startswith("module."):
            k = k[len("module."):]
        if k.startswith("model."):
            k = k[len("model."):]
        out[k] = v
    return out


def infer_vocab_hidden(sd: dict) -> tuple[int, int]:
    # CastedEmbedding commonly stores embedding_weight
    if "embed_tokens.embedding_weight" in sd:
        w = sd["embed_tokens.embedding_weight"]
    elif "embed_tokens.weight" in sd:
        w = sd["embed_tokens.weight"]
    else:
        cands = [(k, v) for k, v in sd.items() if "embed_tokens" in k and torch.is_tensor(v) and v.ndim == 2]
        if not cands:
            raise RuntimeError("Could not find token embedding weights (embed_tokens.*) in checkpoint.")
        _, w = sorted(cands, key=lambda kv: kv[1].numel(), reverse=True)[0]

    vocab_size, hidden_size = w.shape  # (V, D)
    return int(vocab_size), int(hidden_size)


def infer_puzzle_emb(sd: dict):
    # Find a 2D tensor under puzzle_emb that looks like (num_ids, ndim)
    cands = []
    for k, v in sd.items():
        if "puzzle_emb" in k and torch.is_tensor(v) and v.ndim == 2:
            cands.append((k, v))
    if not cands:
        return None, 1, 0  # no puzzle embeddings
    k, t = sorted(cands, key=lambda kv: kv[1].numel(), reverse=True)[0]
    num_ids, ndim = t.shape
    return k, int(num_ids), int(ndim)


def infer_L_layers(sd: dict):
    idxs = set()
    for k in sd.keys():
        if k.startswith("core.layers."):
            rest = k[len("core.layers."):]
            parts = rest.split(".")
            if parts and parts[0].isdigit():
                idxs.add(int(parts[0]))
    return (max(idxs) + 1) if idxs else None


def infer_pos_encodings(sd: dict, default="none"):
    if any(k.startswith("embed_pos.") for k in sd.keys()):
        return "learned"
    if any("rotary_emb" in k for k in sd.keys()):
        return "rope"
    return default


def infer_mlp_t(sd: dict, default=False):
    # If we see mlp_t params in the checkpoint, it's enabled.
    return any(".mlp_t." in k for k in sd.keys()) or bool(default)


def infer_seq_width_from_mlp_t(sd: dict) -> int | None:
    """
    IMPORTANT:
    - For SwiGLU, 'gate_up_proj.weight' has shape [2*intermediate, in_features]
      where in_features == seq_width (what we want).
    - 'down_proj.weight' has in_features == intermediate (NOT what we want).
    So we MUST prefer gate_up_proj.in_features.
    """
    for k, v in sd.items():
        if ".mlp_t." in k and "gate_up_proj.weight" in k and torch.is_tensor(v) and v.ndim == 2:
            return int(v.shape[1])  # in_features
    # Fallback: pick the MIN in_features among mlp_t weights (usually the input width)
    cands = []
    for k, v in sd.items():
        if ".mlp_t." in k and k.endswith(".weight") and torch.is_tensor(v) and v.ndim == 2:
            cands.append(int(v.shape[1]))
    return min(cands) if cands else None


def pad_to_model_batch(batch: dict, target_B: int) -> tuple[dict, int]:
    """
    If model was trained with batch-sized sparse-embedding buffers, forward() may
    assume cfg.batch_size. We pad smaller eval batches up to cfg.batch_size,
    then slice outputs back.
    """
    inputs = batch["inputs"]
    puzzle_ids = batch["puzzle_identifiers"]
    B = inputs.shape[0]
    if B == target_B:
        return batch, B
    if B > target_B:
        return batch, B  # microbatch outside

    pad = target_B - B
    rep_idx = torch.arange(pad, device=inputs.device) % B
    inputs_pad = torch.cat([inputs, inputs.index_select(0, rep_idx)], dim=0)
    puzzle_pad = torch.cat([puzzle_ids, puzzle_ids.index_select(0, rep_idx)], dim=0)
    return {"inputs": inputs_pad, "puzzle_identifiers": puzzle_pad}, B


def accuracy_per_sample(preds: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
    """
    preds/labels: [B, S]
    returns: [B] percent accuracy, ignoring IGNORE_LABEL_ID tokens
    """
    labels = labels.to(preds.device)
    mask = (labels != IGNORE_LABEL_ID)
    correct = (preds == labels) & mask
    denom = mask.sum(dim=1).clamp_min(1)
    return (correct.sum(dim=1).float() / denom.float()) * 100.0


# =========================
# MAIN
# =========================
def run_statistical_eval():
    print("🚀 STARTING BATCHED STATISTICAL EVALUATION")
    print(f"   > Num samples: {NUM_SAMPLES}")
    print(f"   > Eval batch:  {EVAL_BATCH}")
    print(f"   > Configs:     {TEST_CONFIGS}")
    print(f"   > Device:      {DEVICE}")

    # 1) Load data first to get seq_len (avoid hardcoding 81 incorrectly)
    print("📂 Loading dataset arrays...")
    inputs_np = np.load(os.path.join(DATA_PATH, "all__inputs.npy"), mmap_mode="r")
    labels_np = np.load(os.path.join(DATA_PATH, "all__labels.npy"), mmap_mode="r")

    inputs_slice = np.array(inputs_np[:NUM_SAMPLES])
    labels_slice = np.array(labels_np[:NUM_SAMPLES])

    if inputs_slice.ndim != 2:
        raise RuntimeError(f"Expected inputs [N, S], got {inputs_slice.shape}")
    seq_len = int(inputs_slice.shape[1])
    print(f"   > Detected seq_len from data: {seq_len}")

    dataset = TensorDataset(
        torch.tensor(inputs_slice, dtype=torch.long),
        torch.tensor(labels_slice, dtype=torch.long),
    )
    loader = DataLoader(dataset, batch_size=EVAL_BATCH, shuffle=False, drop_last=False)

    # 2) Load checkpoint
    weight_path = os.path.join(CHECKPOINT_DIR, WEIGHTS_FILE)
    print(f"📂 Loading checkpoint: {weight_path}")
    ckpt = torch.load(weight_path, map_location="cpu")

    raw_sd = ckpt.get("model_state", ckpt)
    sd = strip_prefixes(raw_sd)

    # 3) Pull arch config (prefer checkpoint config, fallback to latest_config.yaml)
    arch = {}
    if isinstance(ckpt.get("config", None), dict) and "arch" in ckpt["config"]:
        arch = dict(ckpt["config"]["arch"])
        arch.pop("name", None)
        arch.pop("loss", None)
    else:
        latest_cfg = os.path.join(CHECKPOINT_DIR, "latest_config.yaml")
        if os.path.exists(latest_cfg):
            with open(latest_cfg, "r") as f:
                y = yaml.safe_load(f)
            arch = dict(y.get("arch", {}))
            arch.pop("name", None)
            arch.pop("loss", None)

    # 4) Infer critical shapes from checkpoint
    vocab_size, hidden_size = infer_vocab_hidden(sd)
    pz_key, num_puzzle_identifiers, puzzle_emb_ndim = infer_puzzle_emb(sd)

    pos_enc = infer_pos_encodings(sd, default=str(arch.get("pos_encodings", "none")))
    mlp_t = infer_mlp_t(sd, default=bool(arch.get("mlp_t", False)))

    # ✅ Correct inference for puzzle_emb_len when mlp_t is enabled:
    # seq_width = seq_len + puzzle_emb_len
    puzzle_emb_len = int(arch.get("puzzle_emb_len", 0))
    if mlp_t:
        seq_width_ckpt = infer_seq_width_from_mlp_t(sd)
        if seq_width_ckpt is None:
            raise RuntimeError("mlp_t detected but could not infer seq_width from checkpoint (mlp_t.gate_up_proj.weight).")
        inferred_puzzle_len = int(seq_width_ckpt - seq_len)
        if inferred_puzzle_len < 0:
            raise RuntimeError(f"Inferred negative puzzle_emb_len ({inferred_puzzle_len}). seq_width={seq_width_ckpt}, seq_len={seq_len}")
        puzzle_emb_len = inferred_puzzle_len

    # If puzzle embeddings exist but puzzle_emb_len still 0 (non-mlp_t), pick a sensible default
    if puzzle_emb_ndim > 0 and puzzle_emb_len == 0:
        puzzle_emb_len = 16

    inferred_L_layers = infer_L_layers(sd)
    if "L_layers" not in arch and inferred_L_layers is not None:
        arch["L_layers"] = inferred_L_layers

    # batch_size: keep training batch size if available (pads eval batches safely)
    train_global_bs = None
    if isinstance(ckpt.get("config", None), dict):
        train_global_bs = ckpt["config"].get("global_batch_size", None)
    model_batch_size = int(train_global_bs) if train_global_bs is not None else EVAL_BATCH

    # Ensure required fields exist (reasonable defaults)
    arch.setdefault("num_heads", 8)
    arch.setdefault("expansion", 4)
    arch.setdefault("H_cycles", 1)
    arch.setdefault("L_cycles", 3)
    arch.setdefault("L_layers", inferred_L_layers if inferred_L_layers is not None else 2)
    arch.setdefault("forward_dtype", "bfloat16")

    print("\n🧠 INFERRED / SELECTED MODEL SPECS")
    print(f"   > vocab_size:             {vocab_size}")
    print(f"   > hidden_size:            {hidden_size}")
    print(f"   > seq_len (data):         {seq_len}")
    print(f"   > puzzle_emb_ndim:        {puzzle_emb_ndim} ({'found' if pz_key else 'none'})")
    print(f"   > num_puzzle_identifiers: {num_puzzle_identifiers}")
    print(f"   > puzzle_emb_len:         {puzzle_emb_len}")
    print(f"   > pos_encodings:          {pos_enc}")
    print(f"   > mlp_t:                  {mlp_t}")
    print(f"   > model cfg.batch_size:   {model_batch_size}  (eval batches can be smaller; we pad safely)\n")

    # 5) Build GRM config dict and instantiate
    model_cfg = dict(arch)
    model_cfg.update(
        batch_size=model_batch_size,
        seq_len=seq_len,
        vocab_size=vocab_size,
        hidden_size=hidden_size,
        num_puzzle_identifiers=num_puzzle_identifiers,
        puzzle_emb_ndim=puzzle_emb_ndim,
        puzzle_emb_len=puzzle_emb_len,
        pos_encodings=pos_enc,
        mlp_t=mlp_t,
        causal=False,
    )

    model = GRM(model_cfg)

    # 6) Load weights (now shapes should match)
    model.load_state_dict(sd, strict=True)
    model.to(DEVICE)
    model.eval()
    print("✅ Model weights loaded successfully.\n")

    # 7) Eval
    # results key will be string "H={h}_L={l}"
    results_map = {}
    config_keys = []

    for h, l in TEST_CONFIGS:
        k = f"H={h}, L={l}"
        config_keys.append(k)
        results_map[k] = []

    print(f"⚡ Processing {len(loader)} batches...")
    pbar = tqdm(loader, desc="Evaluating", unit="batch")

    # Use inference_mode; your GRM forward uses enable_grad/no_grad internally
    with torch.inference_mode():
        for batch_inputs, batch_labels in pbar:
            batch_inputs = batch_inputs.to(DEVICE)
            batch_labels = batch_labels.to(DEVICE)

            puzzle_ids = torch.zeros((batch_inputs.shape[0],), dtype=torch.long, device=DEVICE)

            target_B = int(getattr(model.cfg, "batch_size", batch_inputs.shape[0]))
            B = batch_inputs.shape[0]

            batch_stats_str = []

            # Loop over configs for this batch
            for (h_cyc, l_cyc) in TEST_CONFIGS:
                key = f"H={h_cyc}, L={l_cyc}"

                # *** DYNAMICALLY UPDATE L_CYCLES ***
                model.cfg.L_cycles = int(l_cyc)

                acc_list = []
                start = 0
                while start < B:
                    end = min(start + target_B, B)
                    micro_inp = batch_inputs[start:end]
                    micro_lbl = batch_labels[start:end]
                    micro_ids = puzzle_ids[start:end]

                    micro_batch = {"inputs": micro_inp, "puzzle_identifiers": micro_ids}

                    # pad if needed
                    padded_batch, orig_B = pad_to_model_batch(micro_batch, target_B)
                    carry = model.initial_carry(padded_batch)

                    _, out = model(
                        carry=carry,
                        batch=padded_batch,
                        override_H_cycles=int(h_cyc),
                        debug_structure=False,
                        alpha=1.0,
                    )
                    logits = out["logits"][:orig_B]  # slice back

                    preds = torch.argmax(logits, dim=-1)
                    acc = accuracy_per_sample(preds, micro_lbl).detach().cpu().numpy()
                    acc_list.append(acc)

                    start = end

                batch_accs = np.concatenate(acc_list, axis=0)
                results_map[key].extend(batch_accs.tolist())

                current_mean = float(np.mean(results_map[key])) if len(results_map[key]) else 0.0
                batch_stats_str.append(f"{key}:{current_mean:.1f}%")

            pbar.set_postfix_str(" | ".join(batch_stats_str))

    print("\n✅ Done!\n")

    # 8) Summary
    stats = []
    for (h, l) in TEST_CONFIGS:
        key = f"H={h}, L={l}"
        accs = np.array(results_map[key], dtype=float)

        # Check if list is empty
        if accs.size == 0:
            continue

        stats.append({
            "Config": key,
            "H Cycles": h,
            "L Cycles": l,
            "Mean Cell Accuracy": f"{accs.mean():.2f}%",
            "Perfect Solve Rate": f"{(accs == 100.0).mean() * 100.0:.2f}%",
            "Min Acc": f"{accs.min():.2f}%",
            "Max Acc": f"{accs.max():.2f}%",
            "N": int(accs.shape[0]),
        })

    print("📊 STATISTICAL SUMMARY")
    print(pd.DataFrame(stats).to_string(index=False))


if __name__ == "__main__":
    run_statistical_eval()